# 📊 INF01090 - Ciência de Dados - Regression Techniques

**House Prices - Advanced Regression Techniques - Kaggle-Style Competition**

House Prices – Advanced Regression Techniques (<https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/>) is a long-running Kaggle competition that challenges participants to predict the sale prices of homes in Ames, Iowa, using advanced regression models. It is one of Kaggle’s most popular “Getting Started” challenges, designed to build practical skills in data cleaning, feature engineering, and predictive modelling.

## Key facts

- **Platform:** Kaggle
- **Launch year:** 2016
- **Original dataset:** 1,460 training and 1,459 test homes
- **Features:** 79 explanatory variables
- **Primary goal:** Predict house sale price


## 📂 Dataset and Objective

The dataset, compiled by Dean De Cock as an update to the classic Boston Housing dataset, describes almost every aspect of residential homes—covering lot size, room counts, materials, neighbourhood, and more. The task is to use these features to predict each property's final sale price, making it a supervised regression problem that blends statistical and machine learning techniques.

The file **`data_description.txt`** has the full description of each column.

## 🛠 Skills and techniques

This competition is widely used to practise:

- **Feature engineering:** handling missing data, encoding categorical variables, transforming skewed features
- **Model building:** experimenting with algorithms such as linear regression, ridge, lasso, random forest, and gradient boosting
- **Evaluation:** models are ranked by **RMSE on log-transformed prices** in our local competition setup


## 🎯 Assignment Goal

Your goal is to build a regression pipeline that predicts **`SalePrice`** for the hidden test set.

This is not only a leaderboard exercise. You should use this assignment to demonstrate that you understand:

- data cleaning for tabular data
- feature encoding and transformation
- regression modeling
- error analysis
- the effect of different design choices on predictive performance


## 📁 Files You Will Receive

You should work only with the files distributed for this lab:

- **`train_student.csv`** — training data with the target column
- **`test_student.csv`** — test data without the target column
- **`submission_template.csv`** — expected format for submission
- **`data_description.txt`** — attribute descriptions

Do **not** use Kaggle's original public test split for submission. The grading app uses a **custom hidden split** created for this class.


## 🏁 Submission and Leaderboard

Submissions are evaluated in the local grading app:

<https://labo5inf01090-huzhzhojtbeqqknm7duabo.streamlit.app/>

The app expects a CSV file with exactly these columns:

```csv
Id,prediction
1461,210000
1462,179500
1463,220000
```

Rules:

- `Id` must match the IDs in **`test_student.csv`**
- `prediction` must contain one numeric prediction per row
- all test rows must be present
- predictions for `SalePrice` should be non-negative


## 📌 What You Must Deliver

Each group must submit:

1. **A prediction file** for the leaderboard  
2. **This notebook** (completed, with code, outputs, and short explanations)  
3. **A short report section in the notebook** explaining:
   - preprocessing choices
   - feature engineering
   - model(s) tested
   - final model used
   - interpretation of the obtained score


## 👥 Group Work

- Work in groups of up to 4
- All members of the group must understand the final solution
- Use a consistent team name in the leaderboard
- The same team may submit multiple times; the leaderboard keeps the best score


## 📊 Grading Criteria

Your grade will not depend only on leaderboard position. The first three places will get additional grade.

Important:
- a top leaderboard score with poor documentation is **not enough**
- a strong notebook with solid methodology can still receive a high grade even if it is not the top-ranked solution


## 🚦 Recommended Workflow

A good workflow for this assignment is:

1. Inspect the training data
2. Identify numeric and categorical variables
3. Handle missing values
4. Encode categorical variables
5. Optionally transform skewed variables
6. Build a baseline regression model
7. Evaluate improvements using validation on the training set
8. Train your final model on the full student training set
9. Predict on `test_student.csv`
10. Submit the predictions to the leaderboard


## ⚠️ Restrictions and Good Practice

- Do not manually inspect or reconstruct the hidden target values
- Do not hard-code predictions
- Do not submit malformed files to probe the scorer
- Do not use the leaderboard as your only validation method

Recommended:
- create your own validation split from `train_student.csv`
- compare models locally before submitting
- submit only meaningful improvements


## 🧪 Suggested Experiments

You may explore ideas such as:

- dropping columns with many missing values
- imputing missing values numerically and categorically
- one-hot encoding categorical variables
- applying `log1p(SalePrice)` during training
- trying different regularization strengths
- comparing linear and non-linear models
- checking whether some features are highly skewed


## 🧭 Starter Checklist

Before your first submission, verify that:

- [ ] `train_student.csv` loads correctly
- [ ] `test_student.csv` has the same predictor columns as expected
- [ ] your preprocessing works for both train and test
- [ ] your model produces one prediction per test row
- [ ] the output file has exactly two columns: `Id`, `prediction`
- [ ] all predictions are numeric
- [ ] all predictions are non-negative


## 🐍 Suggested Notebook Structure

You may organize your work using sections such as:

1. Data loading
2. Exploratory inspection
3. Missing-value handling
4. Feature encoding
5. Train/validation split
6. Baseline model
7. Improved model
8. Final training and test prediction
9. Submission file generation
10. Reflection


In [10]:
import numpy as np
import pandas as pd
#import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

## 1. Load the data

Update the paths if necessary.


In [11]:
train_df = pd.read_csv("C:\\Users\\pedro\\Desktop\\6mestre\\ds\\lab05_files\\lab05_files\\train_student.csv")
test_df = pd.read_csv("C:\\Users\\pedro\\Desktop\\6mestre\\ds\\lab05_files\\lab05_files\\test_student.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (1022, 81)
Test shape: (438, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,136,20,RL,80.0,10400,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal,174000
1,1453,180,RM,35.0,3675,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2006,WD,Normal,145000
2,763,60,FV,72.0,8640,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,6,2010,Con,Normal,215200
3,933,20,RL,84.0,11670,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2007,WD,Normal,320000
4,436,60,RL,43.0,10667,Pave,NaN,IR2,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2009,ConLw,Normal,212000


## 2. Identify target and predictors


In [12]:
target_col = "SalePrice"
id_col = "Id"

X = train_df.drop(columns=[target_col])
y = train_df[target_col].copy()

print("Target summary:")
display(y.describe())


Target summary:


count      1022.000000
mean     181312.692759
std       77617.461005
min       34900.000000
25%      130000.000000
50%      165000.000000
75%      215000.000000
max      745000.000000
Name: SalePrice, dtype: float64

## 3. Build a local validation split

Use this split to compare models **before** submitting to the leaderboard.


In [13]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_valid.shape)


(817, 80) (205, 80)


## 4. Separate numeric and categorical columns


In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

#Null values analysis
Nans = X_train.select_dtypes(include=["number"]).isnull().sum()
print("Numeric features with missing values:")
print(Nans[Nans > 0])
Nans = X_train.select_dtypes(exclude=["number"]).isnull().sum()
print("Categorical features with missing values:")
print(Nans[Nans > 0])

#No duplicated rows exist:
print("duplicated rows:", X_valid.duplicated().sum())
print("duplicated rows:", X_train.duplicated().sum())

#Filling missing numerical values
for col in numeric_features:
    if(X_train[col].isna().sum() > 0):
        X_train[col] = X_train[col].fillna(X_train[col].median())
    if(X_valid[col].isna().sum() > 0):
        X_valid[col] = X_valid[col].fillna(X_valid[col].median())

#Filling missing categorical values
for col in categorical_features:
    if(X_train[col].isna().sum() > 0):
        X_train[col] = X_train[col].fillna("Unknown")
    if(X_valid[col].isna().sum() > 0):
        X_valid[col] = X_valid[col].fillna("Unknown")

#Check outliers
quartiles = X_train[numeric_features].quantile([0.01, 0.99])   
print(quartiles)

#See which values pass the quartiles
for col in numeric_features:
    print(f"Values in {col} below the 1st percentile:")
    print(X_train[X_train[col] < quartiles.loc[0.01, col]][col])
    print(f"Values in {col} above the 99th percentile:")
    print(X_train[X_train[col] > quartiles.loc[0.99, col]][col])

#Encode some categorical variables
X_train["MSZoning"] = X_train["MSZoning"].replace({"RL": 0, "RM": 1, "C (all)": 2, "FV": 3, "RH": 4})
X_valid["MSZoning"] = X_valid["MSZoning"].replace({"RL": 0, "RM": 1, "C (all)": 2, "FV": 3, "RH": 4})
X_train["Street"] = X_train["Street"].replace({"Pave": 0, "Grvl": 1})
X_valid["Street"] = X_valid["Street"].replace({"Pave": 0, "Grvl": 1})
X_train["LotShape"] = X_train["LotShape"].replace({"Reg": 0, "IR1": 1, "IR2": 2, "IR3": 3})
X_valid["LotShape"] = X_valid["LotShape"].replace({"Reg": 0, "IR1": 1, "IR2": 2, "IR3": 3})
X_train["LandContour"] = X_train["LandContour"].replace({"Lvl": 0, "Bnk": 1, "HLS": 2, "Low": 3})
X_valid["LandContour"] = X_valid["LandContour"].replace({"Lvl": 0, "Bnk": 1, "HLS": 2, "Low": 3})
X_train["Utilities"] = X_train["Utilities"].replace({"AllPub": 0, "NoSewa": 1, "NoSeWa": 2, "ELO": 3})
X_valid["Utilities"] = X_valid["Utilities"].replace({"AllPub": 0, "NoSewa": 1, "NoSeWa": 2, "ELO": 3})
X_train["LandSlope"] = X_train["LandSlope"].replace({"Gtl": 0, "Mod": 1, "Sev": 2})
X_valid["LandSlope"] = X_valid["LandSlope"].replace({"Gtl": 0, "Mod": 1, "Sev": 2})
X_train["Neighborhood"] = X_train["Neighborhood"].replace({"CollgCr": 0, "Veenker": 1, "Crawfor": 2, "NoRidge": 3, "Mitchel": 4, "Somerst": 5, "NWAmes": 6, "OldTown": 7, "BrkSide": 8, "Sawyer": 9, "NridgHt": 10, "NAmes": 11, "SawyerW": 12, "IDOTRR": 13, "MeadowV": 14, "Edwards": 15, "Timber": 16, "Gilbert": 17, "StoneBr": 18, "ClearCr": 19, "NPknolls": 20, "Blmngtn": 21, "BrDale": 22, "SWISU": 23, "Blueste": 24})
X_valid["Neighborhood"] = X_valid["Neighborhood"].replace({"CollgCr": 0, "Veenker": 1, "Crawfor": 2, "NoRidge": 3, "Mitchel": 4, "Somerst": 5, "NWAmes": 6, "OldTown": 7, "BrkSide": 8, "Sawyer": 9, "NridgHt": 10, "NAmes": 11, "SawyerW": 12, "IDOTRR": 13, "MeadowV": 14, "Edwards": 15, "Timber": 16, "Gilbert": 17, "StoneBr": 18, "ClearCr": 19, "NPknolls": 20, "Blmngtn": 21, "BrDale": 22, "SWISU": 23, "Blueste": 24})
X_train["Condition1"] = X_train["Condition1"].replace({"Norm": 0, "Feedr": 1, "Artery": 2, "RRAn": 3, "PosN": 4, "RRAe": 5, "PosA": 6, "RRNe": 7, "Ancestr": 8})
X_valid["Condition1"] = X_valid["Condition1"].replace({"Norm": 0, "Feedr": 1, "Artery": 2, "RRAn": 3, "PosN": 4, "RRAe": 5, "PosA": 6, "RRNe": 7, "Ancestr": 8})
X_train["Condition2"] = X_train["Condition2"].replace({"Norm": 0, "Feedr": 1, "Artery": 2, "RRAn": 3, "PosN": 4, "RRAe": 5, "PosA": 6, "RRNe": 7, "Ancestr": 8})
X_valid["Condition2"] = X_valid["Condition2"].replace({"Norm": 0, "Feedr": 1, "Artery": 2, "RRAn": 3, "PosN": 4, "RRAe": 5, "PosA": 6, "RRNe": 7, "Ancestr": 8})
X_train["BldgType"] = X_train["BldgType"].replace({"1Fam": 0, "2fmCon": 1, "Duplex": 2, "TwnhsE": 3, "Twnhs": 4})
X_valid["BldgType"] = X_valid["BldgType"].replace({"1Fam": 0, "2fmCon": 1, "Duplex": 2, "TwnhsE": 3, "Twnhs": 4})
X_train["HouseStyle"] = X_train["HouseStyle"].replace({"1Story": 0, "2Story": 1, "1.5Fin": 2, "SLvl": 3, "SFoyer": 4, "1.5Unf": 5, "2.5Unf": 6, "2.5Fin": 7})
X_valid["HouseStyle"] = X_valid["HouseStyle"].replace({"1Story": 0, "2Story": 1, "1.5Fin": 2, "SLvl": 3, "SFoyer": 4, "1.5Unf": 5, "2.5Unf": 6, "2.5Fin": 7})
X_train["CentralAir"] = X_train["CentralAir"].replace({"Y": 1, "N": 0})
X_valid["CentralAir"] = X_valid["CentralAir"].replace({"Y": 1, "N": 0})

#Check the number of unique values in each categorical feature
for col in categorical_features:
    print(f"{col}: {X_train[col].nunique()} unique values")

#One-hot encode categorical variables
X_train = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
X_valid = pd.get_dummies(X_valid, columns=categorical_features, drop_first=True)

#See the p-values of the features (doesn't work yet)
#X_train_sm = sm.add_constant(X_train)
#model = sm.OLS(y_train, X_train_sm).fit()
#print(model.summary())

#Columns with more than 50% missing values are dropped
#for col in X_train.columns:
#    if(X_train[col].isnull().sum()/len(X_train)) > 0.5:
#        X_train.drop(columns=[col], inplace=True)
#        numeric_features.remove(col)
#        categorical_features.remove(col)
#    if(X_valid[col].isnull().sum()/len(X_valid)) > 0.5:
#        X_valid.drop(columns=[col], inplace=True)
#        numeric_features.remove(col)
#        categorical_features.remove(col)

#print("After dropping columns with >50% missing values:")
#print("Train shape:", X_train.shape)
#print("Valid shape:", X_valid.shape)

#Future ideas: drop more shit, maybe fuse some columns too 

Numeric features: 37
Categorical features: 43
Numeric features with missing values:
LotFrontage    150
MasVnrArea       2
GarageYrBlt     41
dtype: int64
Categorical features with missing values:
Alley           764
MasVnrType      478
BsmtQual         25
BsmtCond         25
BsmtExposure     25
BsmtFinType1     25
BsmtFinType2     25
Electrical        1
FireplaceQu     386
GarageType       41
GarageFinish     41
GarageQual       41
GarageCond       41
PoolQC          813
Fence           655
MiscFeature     783
dtype: int64
duplicated rows: 0
duplicated rows: 0
           Id  MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  \
0.01     9.16        20.0        24.00   2032.16          3.0          3.0   
0.99  1439.84       190.0       149.84  48642.68         10.0          9.0   

      YearBuilt  YearRemodAdd  MasVnrArea  BsmtFinSF1  ...  GarageArea  \
0.01     1900.0        1950.0        0.00        0.00  ...        0.00   
0.99     2009.0        2009.0      759.76     1572

## 5. Create a preprocessing pipeline

You may improve this pipeline as part of the assignment.


In [15]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])


## 6. Baseline model

Start with a simple model.


In [16]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

baseline_model.fit(X_train, y_train)
pred_valid_baseline = baseline_model.predict(X_valid)

rmse_baseline = mean_squared_error(y_valid, pred_valid_baseline) ** 0.5
mae_baseline = mean_absolute_error(y_valid, pred_valid_baseline)
r2_baseline = r2_score(y_valid, pred_valid_baseline)

print("Baseline RMSE:", rmse_baseline)
print("Baseline MAE :", mae_baseline)
print("Baseline R²  :", r2_baseline)


ValueError: A given column is not a column of the dataframe

## 7. Improved model

Try at least one stronger model and compare the result.


In [26]:
improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

improved_model.fit(X_train, y_train)
pred_valid_improved = improved_model.predict(X_valid)

rmse_improved = mean_squared_error(y_valid, pred_valid_improved) ** 0.5
mae_improved = mean_absolute_error(y_valid, pred_valid_improved)
r2_improved = r2_score(y_valid, pred_valid_improved)

print("Improved RMSE:", rmse_improved)
print("Improved MAE :", mae_improved)
print("Improved R²  :", r2_improved)


Improved RMSE: 25955.68398809484
Improved MAE : 17269.506113821135
Improved R²  : 0.891844100922044


## 8. Compare models

Briefly discuss the difference between the baseline and the improved model.


In [9]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Improved"],
    "RMSE": [rmse_baseline, rmse_improved],
    "MAE": [mae_baseline, mae_improved],
    "R2": [r2_baseline, r2_improved],
})

comparison


,Model,RMSE,MAE,R2
0,Baseline,30682.810886,21579.160836,0.848861
1,Improved,25911.333275,17194.490537,0.892213


**Write a short discussion here.**

- Which model performed better?

The Improved model performed better, because it has the minimum errors (lower RMSE and MAE) and the maximun fit with the data (greater R2).

- Was the improvement large or small?

It was a large improvement, about 5% increase in the R2, 15% decrease with the RMSE and 20% decrease in the MAE.

- What might explain the difference?

The fact that the Improved model uses a more sophisticated and complex non-parametric method (Random Forest) while the Baseline model only uses a simpler Linear Regression.

## 9. Train the final model on the full student training set

Choose your final model and fit it using all available labeled data.


In [10]:
final_model = improved_model  # change if needed

final_model.fit(X, y)
test_predictions = final_model.predict(test_df)

submission = pd.DataFrame({
    id_col: test_df[id_col],
    "prediction": np.maximum(test_predictions, 0)  # keep predictions non-negative
})

submission.head()


,Id,prediction
0,893,140478.416667
1,1106,316577.800000
2,414,117722.000000
3,523,152438.516667
4,1037,326923.336667


## 10. Save the submission file


In [12]:
submission.to_csv("C:\\Users\\pedro\\Desktop\\6mestre\\ds\\lab05_files\\lab05_files\\submission.csv", index=False)
print("Saved submission.csv")


Saved submission.csv


## 11. Submit to the leaderboard

Upload `submission.csv` to:

<https://labo5inf01090-huzhzhojtbeqqknm7duabo.streamlit.app/>

After submitting, record your score below.


**Leaderboard score(s):**

- First submission:
- Best submission:
- Final submitted model:


## 12. Final reflection

Write a short final reflection addressing:

- what preprocessing choices were most important
- whether feature engineering helped
- what model worked best for your group
- what you would try next if you had more time


**Write your final reflection here.**


## 📤 Final Deliverables Checklist

Before submitting your work, verify that you are delivering:

- [ ] completed notebook
- [ ] generated submission file
- [ ] leaderboard score recorded
- [ ] short discussion of preprocessing and model choices
- [ ] final reflection
